# Group Eta — Global Power Plant + OWID CO₂ Data Integration

This notebook integrates the Global Power Plant Database (GPPD) with 2019 country-level indicators from the Our World in Data (OWID) CO₂ dataset.

The final dataset preserves the natural **plant-level grain** of the Global Power Plant Database:

**One row = one real power plant**

Country-level installed-capacity features are calculated separately and merged back onto the individual plant records.

The notebook produces one final output:

**`eta_merged_2019.csv`**

Important terminology:

- `capacity_mw` represents **installed generation capacity**.
- `generation_gwh` represents **actual electricity generation**.
- Missing `generation_gwh` values are not interpreted as zero generation.

## 1. Import Libraries and Load Source Data

In [67]:
import pandas as pd
import numpy as np

df_power = pd.read_csv(
    'global_power_plant_database.csv',
    low_memory=False
)

df_co2 = pd.read_csv(
    'owid-co2-data-1.csv',
    low_memory=False
)

print("Power plant dataset:", df_power.shape)
print("CO2 dataset:", df_co2.shape)

Power plant dataset: (34936, 36)
CO2 dataset: (50411, 79)


### 1.1 Source Coverage Check

In [68]:
original_rows = len(df_power)

original_countries = (
    df_power['country']
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
)

missing_plant_ids = df_power['gppd_idnr'].isna().sum()

duplicate_plant_ids = (
    df_power.loc[
        df_power['gppd_idnr'].notna(),
        'gppd_idnr'
    ]
    .duplicated()
    .sum()
)

philippine_rows = (
    original_countries.eq('PHL')
).sum()

print("Original power plant rows:", original_rows)
print("Unique plant IDs:", df_power['gppd_idnr'].nunique())
print("Missing plant IDs:", missing_plant_ids)
print("Duplicate plant IDs:", duplicate_plant_ids)
print("Countries in original dataset:", original_countries.nunique())
print("Philippines present?:", 'PHL' in set(original_countries))
print("Philippine plant rows:", philippine_rows)

Original power plant rows: 34936
Unique plant IDs: 34936
Missing plant IDs: 0
Duplicate plant IDs: 0
Countries in original dataset: 167
Philippines present?: True
Philippine plant rows: 123


## 2. Clean Global Power Plant Database

The Global Power Plant Database is used as the base table for the final merged dataset.

The plant-level grain is preserved throughout the pipeline:

**One row = one real power plant**

Rows are not removed based on missing `generation_gwh` values. The main dataset is based on plant identity, installed capacity, fuel type, geographic information, and other real plant attributes available in the source.

In [69]:
# Preserve the complete plant-level source table
df_plants = df_power.copy()

# Standardize country ISO-3 codes
df_plants['country'] = (
    df_plants['country']
    .astype(str)
    .str.strip()
    .str.upper()
)

# Standardize primary fuel labels
df_plants['primary_fuel'] = (
    df_plants['primary_fuel']
    .astype(str)
    .str.strip()
)

# Ensure installed capacity is numeric
df_plants['capacity_mw'] = pd.to_numeric(
    df_plants['capacity_mw'],
    errors='coerce'
)

# Keep records with the core fields required for the plant-level dataset
df_plants = df_plants.dropna(
    subset=[
        'country',
        'gppd_idnr',
        'primary_fuel',
        'capacity_mw'
    ]
).copy()

# Installed capacity should be positive
df_plants = df_plants[
    df_plants['capacity_mw'] > 0
].copy()

print("Clean plant-level rows:", len(df_plants))
print("Unique plant IDs:", df_plants['gppd_idnr'].nunique())
print("Countries:", df_plants['country'].nunique())
print(
    "Philippines present?:",
    'PHL' in set(df_plants['country'])
)

df_plants.head()

Clean plant-level rows: 34936
Unique plant IDs: 34936
Countries: 167
Philippines present?: True


,country,country_long,name,gppd_idnr,capacity_mw,latitude,longitude,primary_fuel,other_fuel1,other_fuel2,...,estimated_generation_gwh_2013,estimated_generation_gwh_2014,estimated_generation_gwh_2015,estimated_generation_gwh_2016,estimated_generation_gwh_2017,estimated_generation_note_2013,estimated_generation_note_2014,estimated_generation_note_2015,estimated_generation_note_2016,estimated_generation_note_2017
0,AFG,Afghanistan,Kajaki Hydroelectric Power Plant Afghanistan,GEODB0040538,33.0,32.322,65.1190,Hydro,NaN,NaN,...,123.77,162.90,97.39,137.76,119.50,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1
1,AFG,Afghanistan,Kandahar DOG,WKS0070144,10.0,31.670,65.7950,Solar,NaN,NaN,...,18.43,17.48,18.25,17.70,18.29,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE
2,AFG,Afghanistan,Kandahar JOL,WKS0071196,10.0,31.623,65.7920,Solar,NaN,NaN,...,18.64,17.58,19.10,17.62,18.72,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE
3,AFG,Afghanistan,Mahipar Hydroelectric Power Plant Afghanistan,GEODB0040541,66.0,34.556,69.4787,Hydro,NaN,NaN,...,225.06,203.55,146.90,230.18,174.91,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1
4,AFG,Afghanistan,Naghlu Dam Hydroelectric Power Plant Afghanistan,GEODB0040534,100.0,34.641,69.7170,Hydro,NaN,NaN,...,406.16,357.22,270.99,395.38,350.80,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1


## 3. Clean OWID 2019 Country Snapshot

The OWID dataset is reduced to a single 2019 observation per ISO-3 country code.

This creates the country-level table that will later be joined to the plant-level Global Power Plant Database using a many-to-one relationship:

**many power plants → one OWID country record**

In [70]:
df_co2_2019 = df_co2[
    df_co2['iso_code'].notna()
].copy()

# Keep standard ISO-3 country codes only
df_co2_2019 = df_co2_2019[
    df_co2_2019['iso_code'].str.match(
        r'^[A-Z]{3}$',
        na=False
    )
].copy()

# Keep only the 2019 country snapshot
df_co2_2019 = df_co2_2019[
    df_co2_2019['year'].eq(2019)
].copy()

# Rename OWID country name to avoid conflict with
# the GPPD ISO-3 country column
df_co2_2019 = df_co2_2019.rename(
    columns={
        'country': 'owid_country'
    }
)

duplicate_iso_2019 = df_co2_2019.duplicated(
    subset=['iso_code']
).sum()

print("2019 OWID rows:", len(df_co2_2019))
print(
    "Unique ISO-3 countries:",
    df_co2_2019['iso_code'].nunique()
)
print(
    "Duplicate ISO-3 codes:",
    duplicate_iso_2019
)

display(
    df_co2_2019[
        [
            col for col in [
                'owid_country',
                'iso_code',
                'year',
                'population',
                'gdp',
                'co2',
                'co2_per_capita'
            ]
            if col in df_co2_2019.columns
        ]
    ].head()
)

2019 OWID rows: 218
Unique ISO-3 countries: 218
Duplicate ISO-3 codes: 0


,owid_country,iso_code,year,population,gdp,co2,co2_per_capita
269,Afghanistan,AFG,2019,37856125.0,7.308570e+10,10.400,0.275
894,Albania,ALB,2019,2885011.0,3.328975e+10,4.941,1.712
1069,Algeria,DZA,2019,43294551.0,5.899051e+11,182.425,4.214
1344,Andorra,AND,2019,76492.0,NaN,0.491,6.419
1519,Angola,AGO,2019,32375633.0,1.614089e+11,22.210,0.686


## 4. Country-Level Feature Engineering

Country-level installed-capacity metrics are calculated from the plant-level GPPD records.

These grouped tables are temporary feature-engineering tables only.

They will later be merged back onto the individual plant records so that the final dataset remains at the plant level.

In [71]:
# Total installed capacity and plant count per country
country_totals = (
    df_plants
    .groupby(
        'country',
        as_index=False
    )
    .agg(
        total_plant_count=(
            'gppd_idnr',
            'nunique'
        ),
        total_installed_capacity_mw=(
            'capacity_mw',
            'sum'
        )
    )
)

# Installed capacity by country and primary fuel
country_fuel_capacity = (
    df_plants
    .groupby(
        ['country', 'primary_fuel'],
        as_index=False
    )
    .agg(
        plant_count=(
            'gppd_idnr',
            'nunique'
        ),
        capacity_mw=(
            'capacity_mw',
            'sum'
        )
    )
)

# Add total country installed capacity
country_fuel_capacity = country_fuel_capacity.merge(
    country_totals[
        [
            'country',
            'total_installed_capacity_mw'
        ]
    ],
    on='country',
    how='left',
    validate='many_to_one'
)

# Installed-capacity share of each primary fuel
country_fuel_capacity['capacity_share_pct'] = (
    country_fuel_capacity['capacity_mw']
    / country_fuel_capacity['total_installed_capacity_mw']
    * 100
)

print(
    "Country-level rows:",
    len(country_totals)
)

print(
    "Country-fuel combinations:",
    len(country_fuel_capacity)
)

print(
    "Countries represented:",
    country_fuel_capacity['country'].nunique()
)

display(
    country_fuel_capacity.head(20)
)

Country-level rows: 167
Country-fuel combinations: 698
Countries represented: 167


,country,primary_fuel,plant_count,capacity_mw,total_installed_capacity_mw,capacity_share_pct
0,AFG,Gas,1,42.000,300.550,13.974380
1,AFG,Hydro,6,238.550,300.550,79.371153
2,AFG,Solar,2,20.000,300.550,6.654467
3,AGO,Gas,3,163.680,1071.180,15.280345
4,AGO,Hydro,5,770.600,1071.180,71.939357
5,AGO,Oil,6,136.900,1071.180,12.780298
6,ALB,Hydro,7,1431.000,1529.000,93.590582
7,ALB,Other,1,98.000,1529.000,6.409418
8,ARE,Gas,24,29487.000,30327.000,97.230191
9,ARE,Solar,6,840.000,30327.000,2.769809


### 4.1 Build Country Installed-Capacity Features

A temporary one-row-per-country feature table is created.

Fuel-specific installed capacity and installed-capacity shares are pivoted into separate columns.

Examples:

- `Coal_capacity_mw`
- `Coal_share_pct`
- `Gas_capacity_mw`
- `Gas_share_pct`
- `Hydro_capacity_mw`
- `Hydro_share_pct`

These values describe **installed capacity**, not actual electricity generation.

This temporary country-level feature table will later be merged back onto every real power plant.

In [72]:
# Fuel-specific installed capacity
capacity_pivot = (
    country_fuel_capacity
    .pivot(
        index='country',
        columns='primary_fuel',
        values='capacity_mw'
    )
    .fillna(0)
)

capacity_pivot.columns = [
    f"{fuel}_capacity_mw"
    for fuel in capacity_pivot.columns
]

capacity_pivot = capacity_pivot.reset_index()


# Fuel-specific installed-capacity share
share_pivot = (
    country_fuel_capacity
    .pivot(
        index='country',
        columns='primary_fuel',
        values='capacity_share_pct'
    )
    .fillna(0)
)

share_pivot.columns = [
    f"{fuel}_share_pct"
    for fuel in share_pivot.columns
]

share_pivot = share_pivot.reset_index()


# Combine all country-level capacity features
country_capacity_profile = (
    country_totals
    .merge(
        capacity_pivot,
        on='country',
        how='left',
        validate='one_to_one'
    )
    .merge(
        share_pivot,
        on='country',
        how='left',
        validate='one_to_one'
    )
)

print(
    "Country capacity profile shape:",
    country_capacity_profile.shape
)

print(
    "Countries in capacity profile:",
    country_capacity_profile['country'].nunique()
)

display(
    country_capacity_profile.head()
)

Country capacity profile shape: (167, 33)
Countries in capacity profile: 167


,country,total_plant_count,total_installed_capacity_mw,Biomass_capacity_mw,Coal_capacity_mw,Cogeneration_capacity_mw,Gas_capacity_mw,Geothermal_capacity_mw,Hydro_capacity_mw,Nuclear_capacity_mw,...,Hydro_share_pct,Nuclear_share_pct,Oil_share_pct,Other_share_pct,Petcoke_share_pct,Solar_share_pct,Storage_share_pct,Waste_share_pct,Wave and Tidal_share_pct,Wind_share_pct
0,AFG,9,300.550,0.0,0.0,0.0,42.00,0.0,238.55,0.0,...,79.371153,0.000000,0.000000,0.000000,0.0,6.654467,0.0,0.0,0.0,0.000000
1,AGO,14,1071.180,0.0,0.0,0.0,163.68,0.0,770.60,0.0,...,71.939357,0.000000,12.780298,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000
2,ALB,8,1529.000,0.0,0.0,0.0,0.00,0.0,1431.00,0.0,...,93.590582,0.000000,0.000000,6.409418,0.0,0.000000,0.0,0.0,0.0,0.000000
3,ARE,30,30327.000,0.0,0.0,0.0,29487.00,0.0,0.00,0.0,...,0.000000,0.000000,0.000000,0.000000,0.0,2.769809,0.0,0.0,0.0,0.000000
4,ARG,236,32913.079,0.0,4857.4,0.0,13041.84,0.0,9999.71,1763.0,...,30.382177,5.356533,3.643807,4.024115,0.0,1.566854,0.0,0.0,0.0,0.643149


### 4.2 Fossil and Renewable Installed-Capacity Features

Power plant fuels are grouped conservatively into fossil, renewable, and other/unclassified categories.

Fossil fuels:
- Coal
- Gas
- Oil
- Petcoke

Renewable fuels:
- Biomass
- Geothermal
- Hydro
- Solar
- Wind
- Waste
- Wave and Tidal

Nuclear, storage, cogeneration, and unspecified/other fuels are kept under `other_or_unclassified` rather than being incorrectly classified as fossil or renewable.

The resulting variables represent **installed-capacity shares**, not electricity-generation shares.

In [73]:
fossil_fuels = {
    'Coal',
    'Gas',
    'Oil',
    'Petcoke'
}

renewable_fuels = {
    'Biomass',
    'Geothermal',
    'Hydro',
    'Solar',
    'Wind',
    'Waste',
    'Wave and Tidal'
}

df_capacity_groups = df_plants[
    [
        'country',
        'gppd_idnr',
        'capacity_mw',
        'primary_fuel'
    ]
].copy()

df_capacity_groups['capacity_group'] = np.select(
    [
        df_capacity_groups['primary_fuel'].isin(
            fossil_fuels
        ),
        df_capacity_groups['primary_fuel'].isin(
            renewable_fuels
        )
    ],
    [
        'fossil',
        'renewable'
    ],
    default='other_or_unclassified'
)

country_group_capacity = (
    df_capacity_groups
    .groupby(
        ['country', 'capacity_group'],
        as_index=False
    )
    .agg(
        capacity_mw=(
            'capacity_mw',
            'sum'
        )
    )
)

country_group_profile = (
    country_group_capacity
    .pivot(
        index='country',
        columns='capacity_group',
        values='capacity_mw'
    )
    .fillna(0)
    .reset_index()
)

# Make sure all three group columns exist
for group in [
    'fossil',
    'renewable',
    'other_or_unclassified'
]:
    if group not in country_group_profile.columns:
        country_group_profile[group] = 0

country_group_profile = (
    country_group_profile
    .rename(
        columns={
            'fossil':
                'fossil_capacity_mw',
            'renewable':
                'renewable_capacity_mw',
            'other_or_unclassified':
                'other_or_unclassified_capacity_mw'
        }
    )
)

country_group_profile = country_group_profile.merge(
    country_totals[
        [
            'country',
            'total_installed_capacity_mw'
        ]
    ],
    on='country',
    how='left',
    validate='one_to_one'
)

country_group_profile['fossil_share_pct'] = (
    country_group_profile['fossil_capacity_mw']
    / country_group_profile[
        'total_installed_capacity_mw'
    ]
    * 100
)

country_group_profile['renewable_share_pct'] = (
    country_group_profile['renewable_capacity_mw']
    / country_group_profile[
        'total_installed_capacity_mw'
    ]
    * 100
)

country_group_profile[
    'other_or_unclassified_share_pct'
] = (
    country_group_profile[
        'other_or_unclassified_capacity_mw'
    ]
    / country_group_profile[
        'total_installed_capacity_mw'
    ]
    * 100
)

# total_installed_capacity_mw already exists
# in country_capacity_profile, so remove the
# duplicate before combining the feature tables
country_group_profile = country_group_profile.drop(
    columns=['total_installed_capacity_mw']
)

country_capacity_features = (
    country_capacity_profile
    .merge(
        country_group_profile,
        on='country',
        how='left',
        validate='one_to_one'
    )
)

print(
    "Final country feature table:",
    country_capacity_features.shape
)

display(
    country_capacity_features.head()
)

Final country feature table: (167, 39)


,country,total_plant_count,total_installed_capacity_mw,Biomass_capacity_mw,Coal_capacity_mw,Cogeneration_capacity_mw,Gas_capacity_mw,Geothermal_capacity_mw,Hydro_capacity_mw,Nuclear_capacity_mw,...,Storage_share_pct,Waste_share_pct,Wave and Tidal_share_pct,Wind_share_pct,fossil_capacity_mw,other_or_unclassified_capacity_mw,renewable_capacity_mw,fossil_share_pct,renewable_share_pct,other_or_unclassified_share_pct
0,AFG,9,300.550,0.0,0.0,0.0,42.00,0.0,238.55,0.0,...,0.0,0.0,0.0,0.000000,42.000,0.00,258.55,13.974380,86.025620,0.000000
1,AGO,14,1071.180,0.0,0.0,0.0,163.68,0.0,770.60,0.0,...,0.0,0.0,0.0,0.000000,300.580,0.00,770.60,28.060643,71.939357,0.000000
2,ALB,8,1529.000,0.0,0.0,0.0,0.00,0.0,1431.00,0.0,...,0.0,0.0,0.0,0.000000,0.000,98.00,1431.00,0.000000,93.590582,6.409418
3,ARE,30,30327.000,0.0,0.0,0.0,29487.00,0.0,0.00,0.0,...,0.0,0.0,0.0,0.000000,29487.000,0.00,840.00,97.230191,2.769809,0.000000
4,ARG,236,32913.079,0.0,4857.4,0.0,13041.84,0.0,9999.71,1763.0,...,0.0,0.0,0.0,0.643149,19098.529,3087.46,10727.09,58.027172,32.592180,9.380648


## 5. Main Plant-Level 2019 Merge

The cleaned Global Power Plant Database remains the base table.

Country-level installed-capacity features are merged back onto each real power plant using the ISO-3 country code.

The 2019 OWID country snapshot is then joined using a many-to-one relationship:

**many power plants → one country-level OWID record**

Left joins are used so that power plants are not removed simply because a matching OWID record is unavailable.

In [74]:
# Country + primary-fuel features that correspond
# to each plant's own primary fuel
plant_fuel_features = (
    country_fuel_capacity[
        [
            'country',
            'primary_fuel',
            'plant_count',
            'capacity_mw',
            'capacity_share_pct'
        ]
    ]
    .rename(
        columns={
            'plant_count':
                'country_primary_fuel_plant_count',
            'capacity_mw':
                'country_primary_fuel_capacity_mw',
            'capacity_share_pct':
                'country_primary_fuel_capacity_share_pct'
        }
    )
)

# Start from the real plant-level table
eta_merged_2019 = df_plants.copy()

rows_before_feature_merge = len(
    eta_merged_2019
)

# Add country-wide capacity features
eta_merged_2019 = eta_merged_2019.merge(
    country_capacity_features,
    on='country',
    how='left',
    validate='many_to_one'
)

# Add features corresponding to each plant's
# own primary fuel
eta_merged_2019 = eta_merged_2019.merge(
    plant_fuel_features,
    on=[
        'country',
        'primary_fuel'
    ],
    how='left',
    validate='many_to_one'
)

print(
    "Rows before capacity-feature merge:",
    rows_before_feature_merge
)

print(
    "Rows after capacity-feature merge:",
    len(eta_merged_2019)
)

Rows before capacity-feature merge: 34936
Rows after capacity-feature merge: 34936


In [75]:
rows_before_owid_merge = len(
    eta_merged_2019
)

eta_merged_2019 = eta_merged_2019.merge(
    df_co2_2019,
    left_on='country',
    right_on='iso_code',
    how='left',
    validate='many_to_one',
    indicator='_owid_merge'
)

# Keep a simple data-quality flag
eta_merged_2019['owid_2019_matched'] = (
    eta_merged_2019['_owid_merge']
    .eq('both')
)

print(
    "Rows before OWID merge:",
    rows_before_owid_merge
)

print(
    "Rows after OWID merge:",
    len(eta_merged_2019)
)

print(
    "Countries in final plant-level dataset:",
    eta_merged_2019['country'].nunique()
)

print(
    "Philippines present?:",
    'PHL' in set(
        eta_merged_2019['country']
    )
)

Rows before OWID merge: 34936
Rows after OWID merge: 34936
Countries in final plant-level dataset: 167
Philippines present?: True


## 6. Final Validation

The final dataset is validated to confirm that the natural plant-level grain has been preserved.

The checks verify:

- row count before and after the merge,
- unique plant identifiers,
- duplicate plant records,
- number of countries,
- retention of Philippine power plants,
- unmatched OWID country codes,
- and absence of artificial row inflation.

In [76]:
final_rows = len(
    eta_merged_2019
)

unique_final_plant_ids = (
    eta_merged_2019['gppd_idnr']
    .nunique()
)

duplicate_final_plant_ids = (
    eta_merged_2019['gppd_idnr']
    .duplicated()
    .sum()
)

philippine_final_rows = (
    eta_merged_2019['country']
    .eq('PHL')
    .sum()
)

unmatched_plant_rows = (
    eta_merged_2019['_owid_merge']
    .eq('left_only')
    .sum()
)

unmatched_countries = (
    eta_merged_2019.loc[
        eta_merged_2019['_owid_merge']
        .eq('left_only'),
        [
            'country',
            'country_long'
        ]
    ]
    .value_counts()
    .reset_index(
        name='plant_rows'
    )
    .sort_values('country')
)

same_plant_ids = (
    set(df_plants['gppd_idnr'])
    ==
    set(eta_merged_2019['gppd_idnr'])
)

print("Original source rows:", len(df_power))
print("Clean plant rows:", len(df_plants))
print("Final merged rows:", final_rows)
print(
    "Unique final plant IDs:",
    unique_final_plant_ids
)
print(
    "Duplicate final plant IDs:",
    duplicate_final_plant_ids
)
print(
    "Countries in final dataset:",
    eta_merged_2019['country'].nunique()
)
print(
    "Philippines present?:",
    'PHL' in set(
        eta_merged_2019['country']
    )
)
print(
    "Philippine plant rows:",
    philippine_final_rows
)
print(
    "Unmatched OWID plant rows:",
    unmatched_plant_rows
)
print(
    "Plant IDs preserved exactly?:",
    same_plant_ids
)

print("\nUnmatched country ISO codes:")
display(unmatched_countries)


# Hard validation checks
assert len(eta_merged_2019) == len(df_plants), \
    "Row count changed during merge."

assert duplicate_final_plant_ids == 0, \
    "Duplicate plant IDs were introduced."

assert same_plant_ids, \
    "Plant identities changed during the merge."

assert 'PHL' in set(
    eta_merged_2019['country']
), "Philippines was lost from the final dataset."

print(
    "\nAll plant-level integrity checks passed."
)

Original source rows: 34936
Clean plant rows: 34936
Final merged rows: 34936
Unique final plant IDs: 34936
Duplicate final plant IDs: 0
Countries in final dataset: 167
Philippines present?: True
Philippine plant rows: 123
Unmatched OWID plant rows: 9
Plant IDs preserved exactly?: True

Unmatched country ISO codes:


,country,country_long,plant_rows
2,ESH,Western Sahara,1
0,GUF,French Guiana,6
1,KOS,Kosovo,2



All plant-level integrity checks passed.


In [77]:
phl_preview_columns = [
    'country',
    'country_long',
    'name',
    'gppd_idnr',
    'capacity_mw',
    'primary_fuel',
    'latitude',
    'longitude',
    'commissioning_year',
    'total_installed_capacity_mw',
    'country_primary_fuel_capacity_mw',
    'country_primary_fuel_capacity_share_pct',
    'fossil_share_pct',
    'renewable_share_pct',
    'owid_country',
    'year',
    'population',
    'gdp',
    'co2',
    'co2_per_capita'
]

phl_preview_columns = [
    col
    for col in phl_preview_columns
    if col in eta_merged_2019.columns
]

print("Sample Philippine power plants:")

display(
    eta_merged_2019.loc[
        eta_merged_2019['country'].eq('PHL'),
        phl_preview_columns
    ].head(10)
)

Sample Philippine power plants:


,country,country_long,name,gppd_idnr,capacity_mw,primary_fuel,latitude,longitude,commissioning_year,total_installed_capacity_mw,country_primary_fuel_capacity_mw,country_primary_fuel_capacity_share_pct,fossil_share_pct,renewable_share_pct,owid_country,year,population,gdp,co2,co2_per_capita
18314,PHL,Philippines,AGUS 1,WRI1001957,80.0,Hydro,8.0046,124.2868,NaN,20719.3,3401.1,16.415130,68.527412,31.472588,Philippines,2019.0,110804685.0,9.861147e+11,142.601,1.287
18315,PHL,Philippines,AGUS 2,WRI1001958,180.0,Hydro,8.0528,124.2709,NaN,20719.3,3401.1,16.415130,68.527412,31.472588,Philippines,2019.0,110804685.0,9.861147e+11,142.601,1.287
18316,PHL,Philippines,AGUS 4,WRI1001959,158.1,Hydro,8.1336,124.1983,NaN,20719.3,3401.1,16.415130,68.527412,31.472588,Philippines,2019.0,110804685.0,9.861147e+11,142.601,1.287
18317,PHL,Philippines,AGUS 5,WRI1001960,309.0,Hydro,8.1937,124.1915,NaN,20719.3,3401.1,16.415130,68.527412,31.472588,Philippines,2019.0,110804685.0,9.861147e+11,142.601,1.287
18318,PHL,Philippines,AMBUKLAO,WRI1001902,105.0,Hydro,16.4588,120.7452,NaN,20719.3,3401.1,16.415130,68.527412,31.472588,Philippines,2019.0,110804685.0,9.861147e+11,142.601,1.287
18319,PHL,Philippines,ANGAT,WRI1001900,246.0,Hydro,14.9098,121.1603,NaN,20719.3,3401.1,16.415130,68.527412,31.472588,Philippines,2019.0,110804685.0,9.861147e+11,142.601,1.287
18320,PHL,Philippines,Anda,WRI1029959,83.7,Coal,15.2342,120.6109,2016.0,20719.3,8731.3,42.140902,68.527412,31.472588,Philippines,2019.0,110804685.0,9.861147e+11,142.601,1.287
18321,PHL,Philippines,Armenia Solar,WRI1029964,8.8,Solar,15.4605,120.5918,2016.0,20719.3,1196.4,5.774326,68.527412,31.472588,Philippines,2019.0,110804685.0,9.861147e+11,142.601,1.287
18322,PHL,Philippines,Avion,WRI1029960,100.0,Gas,13.7606,121.0460,2016.0,20719.3,3411.0,16.462911,68.527412,31.472588,Philippines,2019.0,110804685.0,9.861147e+11,142.601,1.287
18323,PHL,Philippines,BACMAN,WRI1001889,130.0,Geothermal,13.0547,123.9678,NaN,20719.3,1848.2,8.920186,68.527412,31.472588,Philippines,2019.0,110804685.0,9.861147e+11,142.601,1.287


## 7. Export Final Dataset

Only the final plant-level merged dataset is exported.

The output retains one row per real power plant and combines:

- GPPD plant-level attributes,
- country-level installed-capacity features,
- and 2019 OWID country indicators.

Final output:

**`eta_merged_2019.csv`**

In [78]:
# Remove the temporary pandas merge indicator.
# The boolean owid_2019_matched column remains.
eta_merged_2019 = eta_merged_2019.drop(
    columns=['_owid_merge']
)

eta_merged_2019.to_csv(
    'eta_merged_2019.csv',
    index=False
)

print(
    "Final dataset saved:",
    eta_merged_2019.shape
)

print(
    "Final row count:",
    len(eta_merged_2019)
)

print(
    "Final unique plant IDs:",
    eta_merged_2019['gppd_idnr'].nunique()
)

print(
    "Output file: eta_merged_2019.csv"
)

from google.colab import files

files.download(
    'eta_merged_2019.csv'
)

Final dataset saved: (34936, 157)
Final row count: 34936
Final unique plant IDs: 34936
Output file: eta_merged_2019.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Appendix — Optional Observed Generation Coverage

The GPPD also contains annual `generation_gwh` fields for 2013–2019.

These values are not used as a requirement for inclusion in the final plant-level dataset because their geographic coverage is limited.

Missing `generation_gwh` values remain missing and are never converted to zero.

This appendix is exploratory only and does not create another output CSV.

In [79]:
generation_cols = [
    col
    for col in df_power.columns
    if col.startswith(
        'generation_gwh_'
    )
]

df_generation_long = df_power.melt(
    id_vars=[
        'country',
        'gppd_idnr'
    ],
    value_vars=generation_cols,
    var_name='generation_year',
    value_name='generation_gwh'
)

df_generation_long['year'] = (
    df_generation_long[
        'generation_year'
    ]
    .str.extract(
        r'(\d{4})'
    )
    .astype(int)
)

# Keep only genuinely observed generation values
df_generation_observed = (
    df_generation_long
    .dropna(
        subset=['generation_gwh']
    )
    .copy()
)

generation_country_coverage = (
    df_generation_observed
    .groupby('year')['country']
    .nunique()
)

print(
    "Countries with observed generation by year:"
)

print(
    generation_country_coverage
)

print(
    "\nPhilippines has observed plant-level generation?:",
    'PHL' in set(
        df_generation_observed['country']
    )
)

Countries with observed generation by year:
year
2013     5
2014     5
2015    29
2016    30
2017    29
2018     4
2019     1
Name: country, dtype: int64

Philippines has observed plant-level generation?: False
